# 사출성형 데이터: 제공 가이드 코드 재현

기존 시간순 평가 실험과 분리한 참고 실험이다. PDF 인쇄 쪽수 기준 43–58쪽의 잡음제거 오토인코더와 69–83쪽의 제품별 준지도 학습을 재현한다. 원문 코드의 전처리와 분리 방식을 유지하므로 **이 결과는 미래 생산 데이터에 대한 검증 성능이 아니다.**

원문에서 실행 불가능한 구문은 호환 수정한다: Adam의 lr→learning_rate, Keras predict_proba→predict, RF max_features='auto'→'sqrt', 미정의 without_label 초기화. 난수 seed=42를 추가한다. 명시된 최종 파라미터를 사용하며 방대한 예시 GridSearch는 재실행하지 않는다. SVM은 튜닝 출력과 학습 코드가 달라 실제 학습 코드(C=.001, gamma=.01)를 따른다.

Colab에서는 Drive를 마운트한 뒤 아래 DATA_DIR만 데이터 폴더로 지정한다. CPU 실행 가능. RF 1,200개 트리와 반복 DNN 학습 때문에 기존 소규모 분석보다 오래 걸릴 수 있다.

In [1]:
import os, json, time, hashlib
from pathlib import Path
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, average_precision_score, roc_auc_score)
from IPython.display import display
tf.config.threading.set_intra_op_parallelism_threads(2)
tf.config.threading.set_inter_op_parallelism_threads(2)
tf.keras.utils.set_random_seed(42)
DATA_DIR=Path(os.environ.get('MOLDING_DATA_DIR', '/content/drive/MyDrive/dataset'))
RESULT_DIR=Path(os.environ.get('REFERENCE_RESULTS', 'results'))
RESULT_DIR.mkdir(parents=True, exist_ok=True)
def read(name):
    d=pd.read_csv(DATA_DIR/name)
    return d.loc[:, ~d.columns.str.startswith('Unnamed:')]
rows=[]
def metrics(y, pred, score):
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return dict(TN=int(tn),FP=int(fp),FN=int(fn),TP=int(tp),
        accuracy=accuracy_score(y,pred),precision=precision_score(y,pred,zero_division=0),
        recall=recall_score(y,pred,zero_division=0),F1=f1_score(y,pred,zero_division=0),
        AP=average_precision_score(y,score),ROC_AUC_score=roc_auc_score(y,score),
        guide_ROC_AUC_hard=roc_auc_score(y,pred))
def record(row):
    rows.append(row)
    pd.DataFrame(rows).to_csv(RESULT_DIR/'metrics.csv',index=False)
    display(pd.DataFrame([row]))
def progress(message):
    with (RESULT_DIR/'progress.txt').open('a',encoding='utf-8') as f:
        f.write(time.strftime('%H:%M:%S')+' '+message+'\n')
    print(message,flush=True)
manifest=[]
for name in ['labeled_data.csv','supervised_label_cn7.csv','moldset_labeled_cn7.csv',
             'moldset_labeled_rg3.csv','moldset_unlabeled_cn7.csv','moldset_unlabeled_rg3.csv']:
    d=read(name)
    manifest.append(dict(file=name,rows=len(d),columns=len(d.columns),
                         sha256=hashlib.sha256((DATA_DIR/name).read_bytes()).hexdigest()))
display(pd.DataFrame(manifest))
pd.DataFrame(manifest).to_csv(RESULT_DIR/'input_manifest.csv',index=False)
print('TensorFlow',tf.__version__)


,file,rows,columns,sha256
0,labeled_data.csv,7996,45,e8d375c8e0c2cdfc309baf01ac2203fc2edb6e9728e836...
1,supervised_label_cn7.csv,6736,25,9b1fce387a94a6eeb8ca4e281ada626a5f325a5f177d61...
2,moldset_labeled_cn7.csv,1211,25,f870b0c259297f20d503f2e2739966553e5db5ef1deb0c...
3,moldset_labeled_rg3.csv,1182,25,14aab21476eea02c823ccb00f82800040280d805dbce5a...
4,moldset_unlabeled_cn7.csv,35239,24,110a2d408d6cb8002caf26f50c06d06e42998e067e7087...
5,moldset_unlabeled_rg3.csv,35941,24,ef9b30014ce4d6671a72f5a4c3f286802b71ff08d52f9d...


TensorFlow 2.21.0


## 1. 오토인코더 입력 재현 (인쇄 43–49쪽)

CN7 LH 다음 RH 순서로 합친다. 제공 supervised_label_cn7.csv가 이 처리와 동일한지 값까지 확인한다. 중복 제거하지 않는다. 원문처럼 정상 전체와 불량 전체에 각각 MinMaxScaler.fit_transform을 적용한다. **정답에 따라 다른 변환을 적용하고 평가 정상도 스케일 추정에 사용하므로 배포 가능한 전처리가 아니다.** 이 부분을 고치면 동일 방식 재현이 아니어서 여기서는 그대로 남긴다.

In [2]:
raw=read('labeled_data.csv')
parts=["CN7 W/S SIDE MLD'G LH", "CN7 W/S SIDE MLD'G RH"]
drop=['_id','TimeStamp','PART_FACT_PLAN_DATE','Reason','PART_FACT_SERIAL','PART_NAME',
      'EQUIP_CD','EQUIP_NAME']+[f'Mold_Temperature_{i}' for i in [1,2,5,6,7,8,9,10,11,12]]
# 원본에서 가이드의 650톤-우진2호기에 해당하는 설비 코드 S14
cn7=pd.concat([raw.loc[(raw.EQUIP_CD=='S14') & (raw.PART_NAME==p)].drop(columns=drop)
               for p in parts],ignore_index=True)
cn7['PassOrFail']=cn7.PassOrFail.map({'Y':0,'N':1})
provided=read('supervised_label_cn7.csv')
pd.testing.assert_frame_equal(cn7[provided.columns],provided,check_dtype=False,rtol=1e-6,atol=1e-6)
print('제공 가공 파일과 공통 열 일치. 원문 전처리에 남는 추가 열:', list(cn7.columns.difference(provided.columns)))
normal=cn7.loc[cn7.PassOrFail==0].drop(columns='PassOrFail').to_numpy()
bad=cn7.loc[cn7.PassOrFail==1].drop(columns='PassOrFail').to_numpy()
normal=MinMaxScaler().fit_transform(normal)
bad=MinMaxScaler().fit_transform(bad)
ae_train=normal[:4000]
ae_test=np.concatenate([normal[4000:],bad])
ae_y=np.r_[np.zeros(len(normal)-4000,dtype=int),np.ones(len(bad),dtype=int)]
assert ae_train.shape[0]==4000 and len(ae_y)==2736 and ae_y.sum()==39
print('정상 학습 4000, 평가 정상 2697 / 불량 39, 입력 변수',ae_train.shape[1])


제공 가공 파일과 공통 열 일치. 원문 전처리에 남는 추가 열: ['Barrel_Temperature_7', 'Switch_Over_Position']
정상 학습 4000, 평가 정상 2697 / 불량 39, 입력 변수 26


## 2. 잡음제거 오토인코더 (인쇄 50–58쪽)

Dropout(.3) → 15 ReLU → 5 ReLU → 15 ReLU → 입력 차원 ReLU. 원문 전처리 코드에는 Switch_Over_Position과 Barrel_Temperature_7이 남아 26개 입력이다. 제공 supervised 파일은 두 열을 제외한 24개이며, 여기서는 원문 코드의 열 구성을 따른다. MSE, Adam(.01), batch=30, 최대 30 epoch, validation_split=.2, EarlyStopping(val_loss, patience=7, mode=min). 최적 가중치 복원 옵션은 원문에 없으므로 사용하지 않는다. 학습 복원 MSE의 평균+5×표준편차를 넘으면 불량이다.

In [3]:
tf.keras.utils.set_random_seed(42)
encoder=tf.keras.Sequential([tf.keras.layers.Input(shape=(ae_train.shape[1],)),
    tf.keras.layers.Dropout(.3),tf.keras.layers.Dense(15,activation='relu'),
    tf.keras.layers.Dense(5,activation='relu')])
decoder=tf.keras.Sequential([tf.keras.layers.Input(shape=(5,)),
    tf.keras.layers.Dense(15,activation='relu'),tf.keras.layers.Dense(ae_train.shape[1],activation='relu')])
ae=tf.keras.Sequential([encoder,decoder])
ae.compile(loss='mse',optimizer=tf.keras.optimizers.Adam(learning_rate=.01),metrics=['accuracy'])
h=ae.fit(ae_train,ae_train,batch_size=30,epochs=30,validation_split=.2,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=7,mode='min')],verbose=0)
train_error=np.mean((ae_train-ae.predict(ae_train,verbose=0))**2,axis=1)
threshold=float(train_error.mean()+5*train_error.std())
error=np.mean((ae_test-ae.predict(ae_test,verbose=0))**2,axis=1)
record(dict(track='guide_AE',product='CN7',model='DenoisingAE',epochs=len(h.history['loss']),
            threshold=threshold,**metrics(ae_y,error>threshold,error)))
pd.DataFrame(h.history).to_csv(RESULT_DIR/'ae_history.csv',index=False)
ae.save(RESULT_DIR/'ae.keras')
progress('AE 완료')


,track,product,model,epochs,threshold,TN,FP,FN,TP,accuracy,precision,recall,F1,AP,ROC_AUC_score,guide_ROC_AUC_hard
0,guide_AE,CN7,DenoisingAE,30,0.07699,2695,2,0,39,0.999269,0.95122,1.0,0.975,1.0,1.0,0.999629


AE 완료


## 3. 제품별 준지도 입력과 공통 평가 (인쇄 69–74쪽)

가이드가 저장한 moldset CSV를 그대로 읽는다. 추가 정규화·중복 제거·시간순 분리·가상 라벨 가중치 조정은 하지 않는다. 제품마다 StratifiedShuffleSplit(test_size=.3, random_state=42)을 적용한다. 제공 파일 기준 CN7 평가 불량은 5건, RG3는 8건이다. 가이드 표의 56건·83건과 다르며 표에 사용한 추가 샘플링을 완전히 재구성할 정보는 없다.

원문 ROC-AUC는 이진 예측값으로 계산한다. 이 값은 guide_ROC_AUC_hard로 보존하고, 확률 기반 AUC와 AP를 별도로 덧붙인다. 평가 구간에서 최고 점수인 반복을 고르지 않고 종료 시 모델을 보고한다.

In [2]:
datasets={}
for product in ['cn7','rg3']:
    d=read(f'moldset_labeled_{product}.csv')
    u=read(f'moldset_unlabeled_{product}.csv')
    y=d.PassOrFail.astype(int).to_numpy()
    x=d.drop(columns='PassOrFail')
    assert list(x.columns)==list(u.columns)
    assert np.isfinite(x.to_numpy()).all() and np.isfinite(u.to_numpy()).all()
    tr,te=next(StratifiedShuffleSplit(n_splits=1,test_size=.3,random_state=42).split(x,y))
    datasets[product]=(x.iloc[tr].to_numpy(),y[tr],x.iloc[te].to_numpy(),y[te],u.to_numpy())
    print(product, '학습',len(tr),'평가',len(te),'평가 불량',int(y[te].sum()),'미라벨',len(u))
def select_pseudo(u,score):
    confidence=np.maximum(score,1-score)
    order=pd.DataFrame({'confidence':confidence}).sort_values('confidence',ascending=False).index.to_numpy()
    k=int(len(u)*.1)
    assert k>0
    return order[:k],order[k:]


cn7 학습 847 평가 364 평가 불량 5 미라벨 35239


rg3 학습 827 평가 355 평가 불량 8 미라벨 35941


## 4. SVM / RandomForest / GaussianNB 반복 학습 (인쇄 74–77쪽)

남은 미라벨의 상위 10%를 매번 추가하고 최초 미라벨의 약 90%가 소진되면 종료한다. 원문은 새 가상 라벨을 추가하기 **전에** 학습하므로 종료 직전 추가분은 최종 모델에 사용되지 않는다. 이 순서도 유지한다. RF는 표기된 1,200 trees, depth=10, bootstrap=False, min_samples_split=5를 사용한다. GaussianNB는 별도 파라미터가 없어 기본값이다.

In [5]:
def run_classic(product,name):
    x,y,xt,yt,u=datasets[product]
    x=x.copy();y=y.copy();u=u.copy()
    limit=int(len(u)*.1); logs=[];iteration=0
    while len(u)>=limit:
        iteration+=1
        if name=='SVM':
            model=SVC(C=.001,gamma=.01,kernel='rbf',class_weight={0:100.,1:1.},probability=True,random_state=42)
        elif name=='RandomForest':
            model=RandomForestClassifier(n_estimators=1200,max_depth=10,max_features='sqrt',
                min_samples_leaf=1,min_samples_split=5,bootstrap=False,random_state=42,n_jobs=4)
        else: model=GaussianNB()
        trained=len(y)
        model.fit(x,y)
        score=model.predict_proba(xt)[:,1];pred=model.predict(xt)
        logs.append(dict(iteration=iteration,trained=trained,remaining=len(u),**metrics(yt,pred,score)))
        us=model.predict_proba(u)[:,1]
        chosen,left=select_pseudo(u,us)
        pseudo=model.predict(u[chosen])
        x=np.concatenate([x,u[chosen]]);y=np.concatenate([y,pseudo]);u=u[left]
        pd.DataFrame(logs).to_csv(RESULT_DIR/f'{product}_{name}_iterations.csv',index=False)
        progress(f'{product} {name} 반복 {iteration}: 학습 {trained}, 남은 미라벨 {len(u)}')
    record(dict(track='guide_semi',product=product.upper(),model=name,
                iterations=iteration,trained=trained,assigned_not_refit=len(y)-trained,**metrics(yt,pred,score)))


In [6]:
run_classic('cn7','SVM')

C:\Users\82102\Documents\Codex\2026-09-08\ai-ai-ai-ai-ai-r\work\guide-venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


cn7 SVM 반복 1: 학습 847, 남은 미라벨 31716


C:\Users\82102\Documents\Codex\2026-09-08\ai-ai-ai-ai-ai-r\work\guide-venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


cn7 SVM 반복 2: 학습 4370, 남은 미라벨 28545


C:\Users\82102\Documents\Codex\2026-09-08\ai-ai-ai-ai-ai-r\work\guide-venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


...

C:\Users\82102\Documents\Codex\2026-09-08\ai-ai-ai-ai-ai-r\work\guide-venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


cn7 SVM 반복 21: 학습 31799, 남은 미라벨 3859


C:\Users\82102\Documents\Codex\2026-09-08\ai-ai-ai-ai-ai-r\work\guide-venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


cn7 SVM 반복 22: 학습 32227, 남은 미라벨 3474


,track,product,model,iterations,trained,assigned_not_refit,TN,FP,FN,TP,accuracy,precision,recall,F1,AP,ROC_AUC_score,guide_ROC_AUC_hard
0,guide_semi,CN7,SVM,22,32227,385,359,0,5,0,0.986264,0.0,0.0,0.0,0.472582,0.928691,0.5


In [7]:
run_classic('cn7','RandomForest')

cn7 RandomForest 반복 1: 학습 847, 남은 미라벨 31716


cn7 RandomForest 반복 2: 학습 4370, 남은 미라벨 28545


cn7 RandomForest 반복 3: 학습 7541, 남은 미라벨 25691


cn7 RandomForest 반복 4: 학습 10395, 남은 미라벨 23122


cn7 RandomForest 반복 5: 학습 12964, 남은 미라벨 20810


...

cn7 RandomForest 반복 19: 학습 30794, 남은 미라벨 4763


cn7 RandomForest 반복 20: 학습 31323, 남은 미라벨 4287


cn7 RandomForest 반복 21: 학습 31799, 남은 미라벨 3859


cn7 RandomForest 반복 22: 학습 32227, 남은 미라벨 3474


,track,product,model,iterations,trained,assigned_not_refit,TN,FP,FN,TP,accuracy,precision,recall,F1,AP,ROC_AUC_score,guide_ROC_AUC_hard
0,guide_semi,CN7,RandomForest,22,32227,385,357,2,3,2,0.986264,0.5,0.4,0.444444,0.475646,0.950975,0.697214


In [8]:
run_classic('cn7','GaussianNB')

cn7 GaussianNB 반복 1: 학습 847, 남은 미라벨 31716

cn7 GaussianNB 반복 2: 학습 4370, 남은 미라벨 28545


cn7 GaussianNB 반복 3: 학습 7541, 남은 미라벨 25691


cn7 GaussianNB 반복 4: 학습 10395, 남은 미라벨 23122

cn7 GaussianNB 반복 5: 학습 12964, 남은 미라벨 20810


...

cn7 GaussianNB 반복 19: 학습 30794, 남은 미라벨 4763


cn7 GaussianNB 반복 20: 학습 31323, 남은 미라벨 4287

cn7 GaussianNB 반복 21: 학습 31799, 남은 미라벨 3859

cn7 GaussianNB 반복 22: 학습 32227, 남은 미라벨 3474

,track,product,model,iterations,trained,assigned_not_refit,TN,FP,FN,TP,accuracy,precision,recall,F1,AP,ROC_AUC_score,guide_ROC_AUC_hard
0,guide_semi,CN7,GaussianNB,22,32227,385,294,65,1,4,0.818681,0.057971,0.8,0.108108,0.441209,0.84234,0.809471


In [9]:
run_classic('rg3','SVM')

C:\Users\82102\Documents\Codex\2026-09-08\ai-ai-ai-ai-ai-r\work\guide-venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


rg3 SVM 반복 1: 학습 827, 남은 미라벨 32347


C:\Users\82102\Documents\Codex\2026-09-08\ai-ai-ai-ai-ai-r\work\guide-venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


rg3 SVM 반복 2: 학습 4421, 남은 미라벨 29113


C:\Users\82102\Documents\Codex\2026-09-08\ai-ai-ai-ai-ai-r\work\guide-venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


...

C:\Users\82102\Documents\Codex\2026-09-08\ai-ai-ai-ai-ai-r\work\guide-venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


rg3 SVM 반복 21: 학습 32394, 남은 미라벨 3937


C:\Users\82102\Documents\Codex\2026-09-08\ai-ai-ai-ai-ai-r\work\guide-venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


rg3 SVM 반복 22: 학습 32831, 남은 미라벨 3544


,track,product,model,iterations,trained,assigned_not_refit,TN,FP,FN,TP,accuracy,precision,recall,F1,AP,ROC_AUC_score,guide_ROC_AUC_hard
0,guide_semi,RG3,SVM,22,32831,393,347,0,8,0,0.977465,0.0,0.0,0.0,0.02805,0.538184,0.5


In [10]:
run_classic('rg3','RandomForest')

rg3 RandomForest 반복 1: 학습 827, 남은 미라벨 32347


rg3 RandomForest 반복 2: 학습 4421, 남은 미라벨 29113


rg3 RandomForest 반복 3: 학습 7655, 남은 미라벨 26202


rg3 RandomForest 반복 4: 학습 10566, 남은 미라벨 23582


rg3 RandomForest 반복 5: 학습 13186, 남은 미라벨 21224


...

rg3 RandomForest 반복 19: 학습 31370, 남은 미라벨 4859


rg3 RandomForest 반복 20: 학습 31909, 남은 미라벨 4374


rg3 RandomForest 반복 21: 학습 32394, 남은 미라벨 3937


rg3 RandomForest 반복 22: 학습 32831, 남은 미라벨 3544


,track,product,model,iterations,trained,assigned_not_refit,TN,FP,FN,TP,accuracy,precision,recall,F1,AP,ROC_AUC_score,guide_ROC_AUC_hard
0,guide_semi,RG3,RandomForest,22,32831,393,341,6,8,0,0.960563,0.0,0.0,0.0,0.029681,0.538545,0.491354


In [11]:
run_classic('rg3','GaussianNB')

rg3 GaussianNB 반복 1: 학습 827, 남은 미라벨 32347


rg3 GaussianNB 반복 2: 학습 4421, 남은 미라벨 29113


rg3 GaussianNB 반복 3: 학습 7655, 남은 미라벨 26202


rg3 GaussianNB 반복 4: 학습 10566, 남은 미라벨 23582


rg3 GaussianNB 반복 5: 학습 13186, 남은 미라벨 21224


...

rg3 GaussianNB 반복 19: 학습 31370, 남은 미라벨 4859


rg3 GaussianNB 반복 20: 학습 31909, 남은 미라벨 4374


rg3 GaussianNB 반복 21: 학습 32394, 남은 미라벨 3937


rg3 GaussianNB 반복 22: 학습 32831, 남은 미라벨 3544


,track,product,model,iterations,trained,assigned_not_refit,TN,FP,FN,TP,accuracy,precision,recall,F1,AP,ROC_AUC_score,guide_ROC_AUC_hard
0,guide_semi,RG3,GaussianNB,22,32831,393,10,337,0,8,0.050704,0.023188,1.0,0.045326,0.020818,0.415346,0.514409


## 5. Keras DNN 반복 학습 (인쇄 78쪽)

24→32→64→Dropout(.25)→32→Dropout(.2)→16→1(sigmoid), Adam 기본값, binary_crossentropy. 반복 사이에 가중치를 유지한다. 매 반복 최대 100 epoch, validation_split=.3, 기본 batch=32. 원문의 precision과 recall이 동시에 이전 값 이상이면 patience를 초기화하는 로직을 그대로 사용한다. 따라서 둘 다 0인 경우에도 100 epoch까지 진행할 수 있다. threshold=.5로 가상 라벨을 생성한다.

In [3]:
def run_dnn(product):
    tf.keras.backend.clear_session();tf.keras.utils.set_random_seed(42)
    x,y,xt,yt,u=datasets[product]
    x=x.astype('float32');u=u.astype('float32');xt=xt.astype('float32');y=y.copy()
    model=tf.keras.Sequential([tf.keras.layers.Input(shape=(24,)),
        tf.keras.layers.Dense(32,activation='relu'),tf.keras.layers.Dense(64,activation='relu'),
        tf.keras.layers.Dropout(.25),tf.keras.layers.Dense(32,activation='relu'),
        tf.keras.layers.Dropout(.2),tf.keras.layers.Dense(16,activation='relu'),
        tf.keras.layers.Dense(1,activation='sigmoid')])
    model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy',
        tf.keras.metrics.Precision(name='precision'),tf.keras.metrics.Recall(name='recall')])
    limit=int(len(u)*.1);iteration=0;logs=[]
    while len(u)>=limit:
        iteration+=1;vp=0;vr=0;cnt=0;trained=len(y)
        for epoch in range(100):
            h=model.fit(x,y,epochs=1,validation_split=.3,verbose=0).history
            if cnt>=10: break
            if h['val_precision'][0]>=vp and h['val_recall'][0]>=vr:
                vp=h['val_precision'][0];vr=h['val_recall'][0];cnt=0
            else: cnt+=1
            if (epoch+1)%20==0: progress(f'{product} DNN 반복 {iteration}, epoch {epoch+1}')
        score=model.predict(xt,verbose=0).ravel();pred=(score>=.5).astype(int)
        logs.append(dict(iteration=iteration,epochs=epoch+1,trained=trained,**metrics(yt,pred,score)))
        us=model.predict(u,verbose=0).ravel();chosen,left=select_pseudo(u,us)
        pseudo=(model.predict(u[chosen],verbose=0).ravel()>=.5).astype(int)
        x=np.concatenate([x,u[chosen]]);y=np.concatenate([y,pseudo]);u=u[left]
        pd.DataFrame(logs).to_csv(RESULT_DIR/f'{product}_DNN_iterations.csv',index=False)
        # 장시간 실행의 진행 보존. 학습 설정 및 종료 규칙은 바꾸지 않는다.
        model.save(RESULT_DIR/f'{product}_dnn_checkpoint.keras')
        np.savez_compressed(RESULT_DIR/f'{product}_dnn_checkpoint_arrays.npz',
                            x=x,y=y,u=u,iteration=iteration)
        progress(f'{product} DNN 반복 {iteration} 완료: 남은 미라벨 {len(u)}')
    model.save(RESULT_DIR/f'{product}_dnn.keras')
    record(dict(track='guide_semi',product=product.upper(),model='DNN',iterations=iteration,
                trained=trained,assigned_not_refit=len(y)-trained,**metrics(yt,pred,score)))


In [13]:
run_dnn('cn7')

cn7 DNN 반복 1, epoch 20


cn7 DNN 반복 1, epoch 40


cn7 DNN 반복 1, epoch 60


cn7 DNN 반복 1, epoch 80


cn7 DNN 반복 1, epoch 100


...

cn7 DNN 반복 3 완료: 남은 미라벨 25691


cn7 DNN 반복 4, epoch 20


cn7 DNN 반복 4, epoch 40


cn7 DNN 반복 4, epoch 60


cn7 DNN 반복 4, epoch 80


In [ ]:
run_dnn('rg3')

## 6. 재현 결과와 해석

아래는 실제 실행 결과이며 원문 성능 표를 복사하지 않는다. AE의 라벨별 정규화와 준지도 학습의 제공 가공 파일·무작위 분리 조건이 기존 시간순 평가와 다르다. 따라서 이전 AP 또는 F1과 빼서 개선 폭이라고 주장할 수 없다. 원문 표와의 표본 수 차이, 원문에 없는 난수 seed 및 라이브러리 버전 차이를 함께 기록한다.

출처: 중소벤처기업부, Korea AI Manufacturing Platform(KAMP), 사출성형기 AI 데이터셋, KAIST(울산과학기술원, ㈜이피엠솔루션즈), 2020.12.14., https://www.kamp-ai.kr/

원본 CSV와 가이드북은 재배포하지 않는다. 연구·공식 활용 시 출처와 활용 내용·문서를 kamp@kaist.ac.kr로 보내라는 제공처 안내가 있다. 이 프로젝트에서 이메일을 발송하지 않았다.

In [ ]:
display(pd.DataFrame(rows).round(6))
progress('전체 재현 완료')